# Config

In [ ]:
%run <Fundraising_Config>

# Imports

In [ ]:
from pyspark.sql.functions import col, when, lit, xxhash64, coalesce, max as spark_max, min as spark_min
from pyspark.sql.types import LongType
from delta.tables import DeltaTable
from datetime import datetime, timedelta
from functools import reduce
from pyspark.sql import DataFrame

# Age Range Segments

In [ ]:
# Load Age Range segments
segment_df = get_gold_table("DimConstituentSegment_AgeRange") \
    .select("ConstituentSegmentKey", "ConstituentSegmentName", "Order")

# Early exit if segment table is empty
if segment_df.count() == 0:
    print("⚠️ DimConstituentSegment_AgeRange is empty - skipping Age Range segmentation. Run Fundraising_GD_CreateSchema first to populate segment definitions.")
else:
    # Load dm_Constituent table
    constituent_df = get_gold_table("dm_Constituent").select("ConstituentKey", "ConstituentName", "Age")

    # Add age boundaries based on segment name
    segment_df = segment_df.withColumn("AgeMin", 
        when(col("ConstituentSegmentName") == "<18", lit(0))
        .when(col("ConstituentSegmentName") == "18-24", lit(18))
        .when(col("ConstituentSegmentName") == "25-34", lit(25))
        .when(col("ConstituentSegmentName") == "35-44", lit(35))
        .when(col("ConstituentSegmentName") == "45-54", lit(45))
        .when(col("ConstituentSegmentName") == "55-64", lit(55))
        .when(col("ConstituentSegmentName") == "65-74", lit(65))
        .when(col("ConstituentSegmentName") == ">75", lit(75))
        .when(col("ConstituentSegmentName") == "Unclassified", lit(None))
    )

    segment_df = segment_df.withColumn("AgeMax", 
        when(col("ConstituentSegmentName") == "<18", lit(17))
        .when(col("ConstituentSegmentName") == "18-24", lit(24))
        .when(col("ConstituentSegmentName") == "25-34", lit(34))
        .when(col("ConstituentSegmentName") == "35-44", lit(44))
        .when(col("ConstituentSegmentName") == "45-54", lit(54))
        .when(col("ConstituentSegmentName") == "55-64", lit(64))
        .when(col("ConstituentSegmentName") == "65-74", lit(74))
        .when(col("ConstituentSegmentName") == ">75", lit(200))
        .when(col("ConstituentSegmentName") == "Unclassified", lit(None))
    )

    # Join constituents with age segments
    classified_df = constituent_df.join(
        segment_df,
        (col("Age").isNotNull()) &
        (col("Age") >= col("AgeMin")) & 
        (col("Age") <= col("AgeMax")),
        "left"
    )

    # Get "Unclassified" segment key
    unclassified_row = get_gold_table("DimConstituentSegment_AgeRange") \
        .filter(col("ConstituentSegmentName") == "Unclassified") \
        .select("ConstituentSegmentKey").first()

    if unclassified_row is None:
        raise ValueError("❌ 'Unclassified' segment not found in DimConstituentSegment_AgeRange")

    unclassified_key = unclassified_row["ConstituentSegmentKey"]

    # Use matched segment or fallback to "Unclassified"
    classified_df = classified_df.withColumn("FinalSegmentKey",
        when(col("ConstituentSegmentKey").isNotNull(), col("ConstituentSegmentKey"))
        .otherwise(lit(unclassified_key))
    )

    # Prepare bridge table records
    new_bridge_df = classified_df.select(
        col("ConstituentKey"),
        col("FinalSegmentKey").alias("ConstituentSegmentKey")
    ).withColumn(
        "ConstituentSegmentBridgeKey",
        xxhash64(col("ConstituentKey"), col("ConstituentSegmentKey")).cast("bigint")
    ).withColumn(
        "ConstituentSegmentMappingId", lit(None).cast("string")
    )

    # Get all segment keys for deletion
    segment_keys = [row["ConstituentSegmentKey"] for row in segment_df.select("ConstituentSegmentKey").collect()]

    # Clear existing Age Range bridge records
    bridge_table = DeltaTable.forName(spark, f"{gold_lakehouse_name}.DimConstituentSegmentBridge_AgeRange")
    bridge_table.delete(f"ConstituentSegmentKey IN ({','.join(map(str, segment_keys))}) AND ConstituentSegmentMappingId IS NULL")

    # Insert new mappings
    new_bridge_df.write.format("delta").mode("append").saveAsTable(f"{gold_lakehouse_name}.DimConstituentSegmentBridge_AgeRange")

    print(f"✅ Age Range segmentation complete: {new_bridge_df.count()} constituent-segment mappings created")

# Lifetime Giving Range Segments

In [ ]:
# Load Lifetime Giving Range segments
segment_df = get_gold_table("DimConstituentSegment_LifetimeGivingRange") \
    .select("ConstituentSegmentKey", "ConstituentSegmentName", "Order")

# Early exit if segment table is empty
if segment_df.count() == 0:
    print("⚠️ DimConstituentSegment_LifetimeGivingRange is empty - skipping Lifetime Giving Range segmentation. Run Fundraising_GD_CreateSchema first to populate segment definitions.")
else:
    # Load dm_Constituent table with lifetime donation amount
    constituent_df = get_gold_table("dm_Constituent").select("ConstituentKey", "ConstituentName", "LifetimeDonationAmount")

    # Replace nulls with 0
    constituent_df = constituent_df.withColumn(
        "LifetimeDonationAmount",
        coalesce(col("LifetimeDonationAmount"), lit(0))
    )

    # Assign numeric ranges based on segment names
    segment_df = segment_df.withColumn("AmountMin", 
        when(col("ConstituentSegmentName") == "<$250", lit(0))
        .when(col("ConstituentSegmentName") == "$250–$999", lit(250))
        .when(col("ConstituentSegmentName") == "$1,000–$4,999", lit(1000))
        .when(col("ConstituentSegmentName") == "$5,000–$9,999", lit(5000))
        .when(col("ConstituentSegmentName") == "$10,000–$24,999", lit(10000))
        .when(col("ConstituentSegmentName") == "$25,000–$49,999", lit(25000))
        .when(col("ConstituentSegmentName") == "$50,000–$99,999", lit(50000))
        .when(col("ConstituentSegmentName") == "$100,000–$499,999", lit(100000))
        .when(col("ConstituentSegmentName") == "$500,000–$999,999", lit(500000))
        .when(col("ConstituentSegmentName") == "$1,000,000+", lit(1000000))
        .when(col("ConstituentSegmentName") == "Unclassified", lit(None))
    )

    segment_df = segment_df.withColumn("AmountMax", 
        when(col("ConstituentSegmentName") == "<$250", lit(249.99))
        .when(col("ConstituentSegmentName") == "$250–$999", lit(999.99))
        .when(col("ConstituentSegmentName") == "$1,000–$4,999", lit(4999.99))
        .when(col("ConstituentSegmentName") == "$5,000–$9,999", lit(9999.99))
        .when(col("ConstituentSegmentName") == "$10,000–$24,999", lit(24999.99))
        .when(col("ConstituentSegmentName") == "$25,000–$49,999", lit(49999.99))
        .when(col("ConstituentSegmentName") == "$50,000–$99,999", lit(99999.99))
        .when(col("ConstituentSegmentName") == "$100,000–$499,999", lit(499999.99))
        .when(col("ConstituentSegmentName") == "$500,000–$999,999", lit(999999.99))
        .when(col("ConstituentSegmentName") == "$1,000,000+", lit(999999999.99))
        .when(col("ConstituentSegmentName") == "Unclassified", lit(None))
    )

    # Join constituents with giving range segments (inner join to only get matched segments)
    classified_df = constituent_df.join(
        segment_df,
        (col("LifetimeDonationAmount") >= col("AmountMin")) & 
        (col("LifetimeDonationAmount") <= col("AmountMax")),
        "inner"
    )

    # Prepare bridge table records (only for constituents that matched a segment)
    new_bridge_df = classified_df.select(
        col("ConstituentKey"),
        col("ConstituentSegmentKey")
    ).withColumn(
        "ConstituentSegmentBridgeKey",
        xxhash64(col("ConstituentKey"), col("ConstituentSegmentKey")).cast("bigint")
    ).withColumn(
        "ConstituentSegmentMappingId", lit(None).cast("string")
    )

    # Get all segment keys for deletion
    segment_keys = [row["ConstituentSegmentKey"] for row in segment_df.select("ConstituentSegmentKey").collect()]

    # Clear existing Lifetime Giving Range bridge records
    bridge_table = DeltaTable.forName(spark, f"{gold_lakehouse_name}.DimConstituentSegmentBridge_LifetimeGivingRange")
    bridge_table.delete(f"ConstituentSegmentKey IN ({','.join(map(str, segment_keys))}) AND ConstituentSegmentMappingId IS NULL")

    # Insert new mappings
    new_bridge_df.write.format("delta").mode("append").saveAsTable(f"{gold_lakehouse_name}.DimConstituentSegmentBridge_LifetimeGivingRange")

    print(f"✅ Lifetime Giving Range segmentation complete: {new_bridge_df.count()} constituent-segment mappings created")

# Gift Recurrence Segments

In [ ]:
from pyspark.sql.functions import col, lit, when, xxhash64, max as spark_max, min as spark_min, coalesce
from pyspark.sql.types import LongType
from delta.tables import DeltaTable
from datetime import datetime, timedelta
from functools import reduce
from pyspark.sql import DataFrame

# Load base tables
config_df = get_gold_table("Configuration")
constituent_df = get_gold_table("dm_Constituent").select("ConstituentKey", "IsNewDonor", "LastDonationDateKey")
donation_df = get_gold_table("FactDonation").select("ConstituentKey", "DonationDateKey", "IsRecurring")
date_df = get_gold_table("DimDate").select("DateKey", "Date", "Year", "FiscalYear")
segment_df = get_gold_table("DimConstituentSegment_GiftRecurrence").select("ConstituentSegmentKey", "ConstituentSegmentName")

# Build segment name-to-key map
segment_key_map = {
    row["ConstituentSegmentName"].strip().lower(): row["ConstituentSegmentKey"]
    for row in segment_df.collect()
}

# Early exit if segment table is empty
if len(segment_key_map) == 0:
    print("⚠️ DimConstituentSegment_GiftRecurrence is empty - skipping Gift Recurrence segmentation. Run Fundraising_GD_CreateSchema first to populate segment definitions.")
else:
    # Date setup
    fiscal_start_month = int(config_df.filter(col("Name") == "FiscalYearStartMonth").select("Value").first()["Value"])
    today = datetime.today()
    one_year_ago = today - timedelta(days=365)
    two_years_ago = today - timedelta(days=730)
    current_year = today.year
    previous_year = current_year - 1

    # Determine fiscal year
    fiscal_today = date_df.filter(col("Date") == lit(today.date())).select("FiscalYear").first()
    fiscal_year = fiscal_today["FiscalYear"] if fiscal_today else current_year
    fiscal_prev_year = fiscal_year - 1

    # Join donations with DimDate
    joined_donations = donation_df.join(date_df, donation_df.DonationDateKey == date_df.DateKey, "left") \
        .select("ConstituentKey", "DonationDateKey", "IsRecurring", "Year", "FiscalYear", "Date")

    # Aggregate metrics
    agg_df = joined_donations.groupBy("ConstituentKey").agg(
        spark_max(when(col("Date") >= lit(one_year_ago), lit(1))).alias("HasRecentDonation"),
        spark_max(when((col("Date") >= lit(two_years_ago)) & (col("Date") < lit(one_year_ago)), lit(1))).alias("HasDonation12to24m"),
        spark_max(when(col("Date") >= lit(two_years_ago), lit(1))).alias("HasDonation24m"),
        spark_min(when(col("Date") < lit(two_years_ago), lit(1))).alias("HasOldDonation"),
        spark_max(when(col("IsRecurring") == True, lit(1))).alias("HasRecurring"),
        spark_max(when(col("Year") == previous_year, lit(1))).alias("HasPrevYearCY"),
        spark_max(when(col("Year") == current_year, lit(1))).alias("HasCurrYearCY"),
        spark_max(when(col("FiscalYear") == fiscal_prev_year, lit(1))).alias("HasPrevYearFY"),
        spark_max(when(col("FiscalYear") == fiscal_year, lit(1))).alias("HasCurrYearFY")
    )

    # Classify into segments
    classified_df = constituent_df.join(agg_df, "ConstituentKey", "left")

    multi_segment_df = classified_df.select(
        "ConstituentKey", "IsNewDonor", "LastDonationDateKey",
        "HasRecentDonation", "HasRecurring", "HasPrevYearCY", "HasCurrYearCY",
        "HasPrevYearFY", "HasCurrYearFY", "HasDonation24m", "HasDonation12to24m", "HasOldDonation"
    )

    segment_rows = []

    # Assign constituents to segments based on conditions
    if "new donor" in segment_key_map:
        segment_rows.append(multi_segment_df.filter(col("IsNewDonor") == True)
                            .withColumn("ConstituentSegmentKey", lit(segment_key_map["new donor"])))

    if "recurring donor" in segment_key_map:
        segment_rows.append(multi_segment_df.filter(col("HasRecurring") == 1)
                            .withColumn("ConstituentSegmentKey", lit(segment_key_map["recurring donor"])))

    if "active" in segment_key_map:
        segment_rows.append(multi_segment_df.filter(col("HasRecentDonation") == 1)
                            .withColumn("ConstituentSegmentKey", lit(segment_key_map["active"])))

    if "lybnt t12m" in segment_key_map:
        # LYBNT T12M: Donated in the previous trailing 12 months (12-24 months ago), but NOT in the last 12 months
        segment_rows.append(multi_segment_df.filter((col("HasDonation12to24m") == 1) & (coalesce(col("HasRecentDonation"), lit(0)) != 1))
                            .withColumn("ConstituentSegmentKey", lit(segment_key_map["lybnt t12m"])))

    if "lybnt cy" in segment_key_map:
        segment_rows.append(multi_segment_df.filter((col("HasPrevYearCY") == 1) & (coalesce(col("HasCurrYearCY"), lit(0)) != 1))
                            .withColumn("ConstituentSegmentKey", lit(segment_key_map["lybnt cy"])))

    if "lybnt fy" in segment_key_map:
        segment_rows.append(multi_segment_df.filter((col("HasPrevYearFY") == 1) & (coalesce(col("HasCurrYearFY"), lit(0)) != 1))
                            .withColumn("ConstituentSegmentKey", lit(segment_key_map["lybnt fy"])))

    if "lapsed donor" in segment_key_map:
        segment_rows.append(multi_segment_df.filter((coalesce(col("HasDonation24m"), lit(0)) != 1) & (col("HasOldDonation") == 1))
                            .withColumn("ConstituentSegmentKey", lit(segment_key_map["lapsed donor"])))

    if "prospect" in segment_key_map:
        segment_rows.append(multi_segment_df.filter(col("LastDonationDateKey").isNull())
                            .withColumn("ConstituentSegmentKey", lit(segment_key_map["prospect"])))

    # Combine all rows - guard against empty segment_rows
    if not segment_rows:
        print("⚠️ No segment conditions matched - skipping bridge table update")
    else:
        union_df = reduce(DataFrame.unionAll, segment_rows)

        # Final bridge DF
        new_bridge_df = union_df.select("ConstituentKey", "ConstituentSegmentKey") \
            .dropDuplicates() \
            .withColumn("ConstituentSegmentBridgeKey", xxhash64(col("ConstituentKey"), col("ConstituentSegmentKey")).cast("bigint")) \
            .withColumn("ConstituentKey", col("ConstituentKey").cast(LongType())) \
            .withColumn("ConstituentSegmentKey", col("ConstituentSegmentKey").cast(LongType())) \
            .withColumn("ConstituentSegmentBridgeKey", col("ConstituentSegmentBridgeKey").cast(LongType()))

        # Remove old values from bridge
        bridge_table = DeltaTable.forName(spark, f"{gold_lakehouse_name}.DimConstituentSegmentBridge_GiftRecurrence")
        bridge_table.delete(f"ConstituentSegmentKey IN ({','.join(map(str, segment_key_map.values()))}) AND ConstituentSegmentMappingId IS NULL")

        # Insert updated bridge records
        new_bridge_df.write \
            .format("delta") \
            .mode("append") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{gold_lakehouse_name}.DimConstituentSegmentBridge_GiftRecurrence")

        print(f"✅ Gift Recurrence segments updated: {new_bridge_df.count()} constituent-segment mappings created")